# Demo 6 — Self-Refine with an EXTERNAL Verifier (ruff + mypy)
## Session 6: Advanced Prompt Engineering — Optional / Extra

> **MLflow as the tape recorder.** Every prompt sent, every response returned, every draft, every critique, every ruff/mypy finding — captured automatically by `mlflow.openai.autolog()` and the `@mlflow.trace` decorators below. Run the cells, then open the MLflow UI to replay every input and output the model saw.

**The setup.** Huang et al. 2023 ([arXiv:2310.01798](https://arxiv.org/abs/2310.01798)) showed something embarrassing: when you ask a model to self-correct *without* an external signal, it gets **worse**. The model is not an oracle on its own correctness. "Is this right?" loops degrade GSM8K accuracy.

**The fix.** Give the critic something the generator *didn't* have. A rubric, a unit test, a linter, a static analyser. Anything. The critic needs ground.

**This demo.** We make a model write a non-trivial Python function. Round 0 will be buggy — type errors, missing edge cases, bare excepts, the whole catalogue. Then we run **ruff + mypy** on the draft, hand the linter output to the critic, and watch the model iteratively fix its own code as the tools yell at it.

Then we run the same loop with the **ungrounded** critic ("do you see any bugs?") and watch it stall or regress. Side-by-side. That's the whole punchline of Misconception 0.

**Citation:** Madaan et al. 2023, *Self-Refine: Iterative Refinement with Self-Feedback* — [arXiv:2303.17651](https://arxiv.org/abs/2303.17651) (NeurIPS 2023). And the cautionary counterpart: Huang et al. 2023 — [arXiv:2310.01798](https://arxiv.org/abs/2310.01798).

**Dependencies:** `openai`, `mlflow >= 3.10`, `pandas`, **`ruff`**, **`mypy`** on the PATH.

```bash
pip install ruff mypy
mlflow server --host 127.0.0.1 --port 5000
```

**Cost when real:** ~$0.02 per run (`gpt-4o-mini`, ~6–10 calls).

## Setup — env-var driven config

Everything below is configured from environment variables. Set `OPENAI_API_KEY` (or `OPENAI_API_KEY`) for live calls; leave it unset for mock mode. Override `MLFLOW_TRACKING_URI`, `MLFLOW_EXPERIMENT`, `OPENAI_BASE_URL`, `MODEL` to point at any OpenAI-compatible endpoint (OpenAI, Azure, vLLM, Ollama, Together, etc.) and any MLflow server.

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
export OPENAI_API_KEY=sk-...
export MODEL=gpt-4o-mini
```

**Local-by-default.** Notebook ships with `OPENAI_BASE_URL=http://localhost:11434/v1` and `MODEL=qwen2.5-coder:14b` (Ollama — Self-Refine code review benefits from a strong generator + critic). Set the env vars to point at OpenAI, Anthropic, vLLM, Together, etc. — any OpenAI-compatible endpoint.

**Cost tracking uses hypothetical local-LLM rates** (~$0.00005/1K tokens) so the cost-aware traces and scorers still produce meaningful numbers in the demos. Override the `PRICING` dict with your real per-1K rates for production.

In [ ]:
import os
import re
import json
import shutil
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Optional

import pandas as pd
import mlflow
from mlflow.entities import SpanType
from openai import OpenAI

# --- MLflow tracking ---------------------------------------------------------
MLFLOW_TRACKING_URI      = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")  # optional (Databricks/remote)
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")  # optional
MLFLOW_REGISTRY_URI      = os.getenv("MLFLOW_REGISTRY_URI", MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT        = os.getenv("MLFLOW_EXPERIMENT", "session6/demo_06_self_refine_lint")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_REGISTRY_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# --- LLM (OpenAI-compatible: OpenAI, Azure, vLLM, Ollama, Together, etc.) ---
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")  # Ollama default
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")  # Ollama ignores; any non-empty string works
MODEL    = os.getenv("MODEL", "qwen2.5-coder:14b")  # Self-Refine benefits from a strong generator + critic

# Reproducibility default. Both generator AND critic stay at 0.0 for the demo —
# the convergence behaviour we want to show is structural, not sampling-noise.
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

# Per-1K-token pricing (USD). Update at each model release.
PRICING = {
    "gpt-4o":             {"in": 0.0025,   "out": 0.010},
    "gpt-4o-mini":        {"in": 0.00015,  "out": 0.0006},
    "o1-mini":            {"in": 0.003,    "out": 0.012},
    "claude-sonnet-4-6":  {"in": 0.003,    "out": 0.015},
    # Hypothetical local-LLM pricing (electricity + amortised GPU per 1K tokens — adjust to your infra)
    "qwen2.5-coder:14b":  {"in": 0.00005,  "out": 0.00005},
    "qwen2.5-coder:7b":   {"in": 0.00002,  "out": 0.00002},
    "llama3.1:8b":        {"in": 0.00002,  "out": 0.00002},
}

def price_of(model: str, usage) -> float:
    """USD cost for a single chat call. 0.0 in mock mode (no usage).
    Unknown models fall back to hypothetical local-LLM rates (~$0.00005/1K)."""
    if usage is None:
        return 0.0
    p = PRICING.get(model, {"in": 0.00005, "out": 0.00005})  # hypothetical fallback
    return usage.prompt_tokens / 1000 * p["in"] + usage.completion_tokens / 1000 * p["out"]


def tag_cost_latency(latency_ms: float, cost_usd: float) -> None:
    """Attach cost + latency to active span and root trace."""
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("latency_ms", latency_ms)
        span.set_attribute("cost_usd", cost_usd)
    try:
        mlflow.update_current_trace(tags={
            "cost_usd":   f"{cost_usd:.6f}",
            "latency_ms": f"{latency_ms:.0f}",
        })
    except Exception:
        pass


client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY or "no-key")

# --- Turn MLflow into the tape recorder -------------------------------------
# Captures every chat.completions.create call — request, response, tokens, latency, cost.
mlflow.openai.autolog()

USE_MOCK = not OPENAI_API_KEY
print(f"MLflow:  {MLFLOW_TRACKING_URI}  (exp: {MLFLOW_EXPERIMENT})")
print(f"LLM:     {OPENAI_BASE_URL}  model={MODEL}  mock={USE_MOCK}  temperature={TEMPERATURE}")
print(f"UI:      open {MLFLOW_TRACKING_URI}  -> Experiments -> {MLFLOW_EXPERIMENT}")
print(f"NOTE: defaults to local Ollama (http://localhost:11434/v1). Set OPENAI_BASE_URL/OPENAI_API_KEY/MODEL to point at OpenAI/Anthropic/vLLM.")

if USE_MOCK:
    print("\nWARNING: no OPENAI_API_KEY / OPENAI_API_KEY set — running in MOCK mode.")
    print("         Drafts/critiques will be pre-canned (buggy R0 -> clean R1).")

# --- Check ruff + mypy are on PATH ------------------------------------------
RUFF = shutil.which("ruff")
MYPY = shutil.which("mypy")
if not RUFF or not MYPY:
    print("\nMISSING TOOL. Install with:  pip install ruff mypy")
    print(f"  ruff: {RUFF or 'NOT FOUND'}")
    print(f"  mypy: {MYPY or 'NOT FOUND'}")
else:
    print(f"ruff: {RUFF}")
    print(f"mypy: {MYPY}")

## The task — deceptively annoying

Write a Python function:

```python
def parse_iso_timestamp(s: str) -> datetime:
    """Parse an ISO-8601-ish timestamp string into a `datetime`.

    Must handle:
      - `Z`-terminated UTC      (e.g. `2025-01-15T10:30:00Z`)
      - explicit offset         (e.g. `2025-01-15T10:30:00+05:30`)
      - naive (no zone)         (e.g. `2025-01-15T10:30:00`)

    Raises `ValueError` on anything else. No bare excepts. No `print`.
    """
```

Three edge cases, one strict rubric. Round-0 drafts from `gpt-4o-mini` will *usually* miss the `Z` → `+00:00` substitution (which `datetime.fromisoformat` requires before 3.11) or smuggle in a bare `except:` to swallow bad input. Either way: ruff and mypy will have opinions.

In [ ]:
# -- The task + the rubric -------------------------------------------------

TASK = '''Write a Python function with this exact signature:

    def parse_iso_timestamp(s: str) -> datetime: ...

It must:
  - Handle `Z`-terminated UTC strings (`2025-01-15T10:30:00Z`).
  - Handle explicit numeric offsets (`2025-01-15T10:30:00+05:30`).
  - Handle naive timestamps with no zone (`2025-01-15T10:30:00`).
  - Raise `ValueError` on any string it cannot parse.

Include the `from datetime import datetime, timezone` import inside the snippet.
Return only the import + function in a single ```python fenced block.'''

RUBRIC = '''Rubric (each rule is a hard constraint):
  1. Signature is exactly `def parse_iso_timestamp(s: str) -> datetime:` — typed.
  2. Handles all three forms: `Z`-UTC, explicit offset, and naive.
  3. Raises `ValueError` (not a bare `Exception`, not silently returns None) on garbage input.
  4. No bare `except:` clauses. Catch specific exceptions only.
  5. Zero ruff findings AND zero mypy findings.'''

print(TASK)
print()
print(RUBRIC)

## The verifier — `ruff` + `mypy` as the external signal

This is the function that does the actual grounding. It writes the candidate to a temp file, runs ruff and mypy as subprocesses, and returns a structured report: `{"n_err": int, "findings": [...]}`.

The critic prompt gets this report verbatim. That's the entire mechanism.

In [ ]:
# -- The lint verifier -----------------------------------------------------

def lint(code: str) -> dict:
    """Run ruff + mypy on `code`. Returns {'n_err': int, 'findings': [str, ...], 'raw': str}."""
    if not RUFF or not MYPY:
        return {"n_err": 0, "findings": ["(ruff/mypy not installed — verifier disabled)"], "raw": ""}

    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code)
        path = f.name

    findings: list[str] = []

    # ruff: JSON output is precise and structured
    ruff = subprocess.run(
        [RUFF, "check", "--output-format=json", "--select", "ALL", "--ignore", "D,ANN101,COM812,ISC001", path],
        capture_output=True, text=True,
    )
    try:
        ruff_items = json.loads(ruff.stdout) if ruff.stdout.strip() else []
    except json.JSONDecodeError:
        ruff_items = []
    for item in ruff_items:
        loc = item.get("location", {})
        findings.append(
            f"ruff {item.get('code', '?')} line {loc.get('row', '?')}: {item.get('message', '')}"
        )

    # mypy: line-oriented, concise
    mypy = subprocess.run(
        [MYPY, "--no-error-summary", "--no-color-output", "--show-error-codes", path],
        capture_output=True, text=True,
    )
    for line in (mypy.stdout + mypy.stderr).splitlines():
        line = line.strip()
        if not line or line.startswith("Success"):
            continue
        # strip the temp path prefix for readability
        line = line.replace(path + ":", "mypy line ")
        findings.append(line)

    Path(path).unlink(missing_ok=True)

    return {
        "n_err": len(findings),
        "findings": findings,
        "raw": "\n".join(findings) if findings else "(no findings — clean)",
    }

# Smoke test the verifier on something deliberately broken
_broken = '''from datetime import datetime
def parse(s):
    try:
        return datetime.fromisoformat(s)
    except:
        pass
'''
report = lint(_broken)
print(f"verifier OK — found {report['n_err']} issues on the broken sample:")
for f in report["findings"][:8]:
    print("  •", f)

## The three prompts

Three roles, three prompts, one model. The **critic** is the only one that sees the lint output — that's what makes the loop *grounded*.

In [ ]:
# -- The three prompts -----------------------------------------------------

GEN_PROMPT = '''{task}'''

CRITIC_PROMPT = '''You are reviewing a Python function against this rubric:

{rubric}

Spec:
{task}

Current draft:
```python
{draft}
```

External verifier output (ruff + mypy on the draft above):
{lint_report}

List every rubric violation. For each, name the rubric rule number and a one-line concrete fix.
Output JSON only:
{{"issues": [{{"rule": <int 1-5>, "fix": "<one line>"}}, ...], "done": <bool — true iff zero issues and zero verifier findings>}}'''

REFINE_PROMPT = '''Apply every fix in the critique to the draft. Preserve correct behaviour.
Return only the revised import + function in a single ```python fenced block.

Spec:
{task}

Previous draft:
```python
{draft}
```

Critique (JSON):
{critique}'''

# -- Ungrounded critic (Huang et al. failure mode) ------------------------
UNGROUNDED_CRITIC_PROMPT = '''Look at this Python function. Do you see any bugs or issues?

```python
{draft}
```

Output JSON only:
{{"issues": [{{"rule": 0, "fix": "<one line>"}}, ...], "done": <bool — true iff you think the code is fine>}}'''

print("prompts ready — GEN, CRITIC (grounded), REFINE, UNGROUNDED_CRITIC")

In [ ]:
# -- LLM call + code extraction helpers -----------------------------------

# Pre-canned mock drafts for offline runs.
_MOCK_ROUND_0 = '''```python
from datetime import datetime

def parse_iso_timestamp(s):
    try:
        return datetime.fromisoformat(s)
    except:
        print("bad timestamp")
        return None
```'''

_MOCK_ROUND_1 = '''```python
from datetime import datetime, timezone

def parse_iso_timestamp(s: str) -> datetime:
    if not isinstance(s, str) or not s:
        raise ValueError(f"expected non-empty str, got {type(s).__name__}")
    candidate = s.replace("Z", "+00:00") if s.endswith("Z") else s
    try:
        return datetime.fromisoformat(candidate)
    except ValueError as exc:
        raise ValueError(f"cannot parse ISO timestamp: {s!r}") from exc
```'''

_MOCK_CRITIC_GROUNDED = '''{"issues": [
  {"rule": 1, "fix": "Add `: str` and `-> datetime` to the signature."},
  {"rule": 2, "fix": "Replace trailing Z with +00:00 before fromisoformat."},
  {"rule": 3, "fix": "Raise ValueError on parse failure instead of returning None."},
  {"rule": 4, "fix": "Replace bare except with `except ValueError as exc`."},
  {"rule": 5, "fix": "Remove the print call (ruff T201) and import timezone."}
], "done": false}'''

_MOCK_CRITIC_UNGROUNDED = '''{"issues": [], "done": true}'''

_mock_state = {"round": 0}

def llm(prompt: str, *, role: str, temperature: float | None = None) -> str:
    """Single OpenAI call. In mock mode, returns a pre-canned response per role.

    Threads TEMPERATURE (default 0.0). Both critic AND generator are deterministic
    for this demo — see the cache-discipline note in the loop cell.
    """
    if temperature is None:
        temperature = TEMPERATURE

    if USE_MOCK:
        tag_cost_latency(0.0, 0.0)
        if role == "gen":
            return _MOCK_ROUND_0
        if role == "refine":
            return _MOCK_ROUND_1
        if role == "critic_grounded":
            return _MOCK_CRITIC_GROUNDED
        if role == "critic_ungrounded":
            return _MOCK_CRITIC_UNGROUNDED
        return "{}"

    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=800,
    )
    latency_ms = (time.time() - t0) * 1000
    cost = price_of(MODEL, resp.usage)
    tag_cost_latency(latency_ms, cost)
    return resp.choices[0].message.content or ""


def extract_code(text: str) -> str:
    """Pull the contents of the first ```python ... ``` block; fall back to raw text."""
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()


def extract_json(text: str) -> dict:
    """Best-effort JSON parse — strip code fences if present."""
    m = re.search(r"```(?:json)?\s*\n(.*?)```", text, re.DOTALL)
    payload = m.group(1).strip() if m else text.strip()
    try:
        return json.loads(payload)
    except json.JSONDecodeError:
        # last-ditch: find the first {...} block
        m2 = re.search(r"\{.*\}", payload, re.DOTALL)
        if m2:
            try:
                return json.loads(m2.group(0))
            except json.JSONDecodeError:
                pass
        return {"issues": [], "done": False, "_parse_error": True}


print("llm / extract_code / extract_json ready.")

## The Self-Refine loop — generate, critique-with-lint, refine

One MLflow `CHAIN` trace per run. Each round wraps three LLM sub-spans (`gen` / `critic` / `refine`) plus the lint subprocess timing. Per-round attributes (`n_err_before`, `n_err_after`, `delta`) make the convergence plot fall out of the UI for free.

**Exit conditions** (in order): `n_err == 0` (converged), `n_err_after >= n_err_before` (plateau or regression), `round == max_rounds` (cap).

In [ ]:
# -- The Self-Refine loop --------------------------------------------------
#
# DO NOT cache intermediate Self-Refine outputs — each round MUST change by design.
# Generator / critic / refine outputs are intentionally NOT memoised. If you wrap
# `llm(...)` with `@lru_cache` or any (prompt, model)-keyed cache, round 2 will
# return round 1's payload and the loop will appear to "converge" by silently
# repeating itself. That's the most common production footgun for this pattern.
#
# Temperature note: critic AND generator both run at TEMPERATURE (default 0.0)
# for this demo so the convergence behaviour is reproducible across re-runs.
# In production, the critic can stay at 0.0 (deterministic verifier-driven critique)
# while the generator/refiner may use a slightly higher temperature to escape local
# minima — but ONLY if you have a verifier (ruff/mypy/pytest) to keep them honest.

@mlflow.trace(
    name="self_refine_lint",
    span_type=SpanType.CHAIN,
    attributes={"verifier": "ruff+mypy", "max_rounds": 3, "rubric_rules": 5},
)
def self_refine(task: str, rubric: str, max_rounds: int = 3, *, grounded: bool = True) -> dict:
    """Run Self-Refine. If grounded=False, the critic does NOT see lint output (Huang failure mode)."""
    _mock_state["round"] = 0

    root_span = mlflow.get_current_active_span()
    if root_span:
        root_span.set_inputs({"task": task, "rubric": rubric,
                              "max_rounds": max_rounds, "grounded": grounded,
                              "model": MODEL, "temperature": TEMPERATURE})

    # ---- Round 0: generate + initial lint ----
    with mlflow.start_span(name="round_0_gen", span_type=SpanType.LLM) as sp:
        gen_prompt = GEN_PROMPT.format(task=task)
        sp.set_inputs({"prompt": gen_prompt,
                       "params": {"model": MODEL, "temperature": TEMPERATURE}})
        # DO NOT cache: each round must produce a fresh draft.
        draft_raw = llm(gen_prompt, role="gen")
        draft = extract_code(draft_raw)
        report = lint(draft)
        sp.set_attribute("n_err_after", report["n_err"])
        sp.set_attribute("grounded", grounded)
        sp.set_outputs({"draft": draft, "raw": draft_raw,
                        "n_err": report["n_err"], "findings": report["findings"]})

    print(f"\n--- Round 0 (initial draft) ---")
    print(draft)
    print(f"\n[verifier] {report['n_err']} finding(s):")
    for f in report["findings"]:
        print("  •", f)

    history = [{"round": 0, "draft": draft, "n_err": report["n_err"], "findings": list(report["findings"])}]
    exit_reason = "max_rounds"

    for r in range(1, max_rounds + 1):
        _mock_state["round"] = r
        n_err_before = report["n_err"]

        # ---- critic ----
        # DO NOT cache: critique depends on the latest draft AND latest lint output.
        with mlflow.start_span(name=f"round_{r}_critic", span_type=SpanType.LLM) as sp:
            if grounded:
                critic_prompt = CRITIC_PROMPT.format(
                    task=task, rubric=rubric, draft=draft,
                    lint_report=report["raw"],
                )
                role = "critic_grounded"
            else:
                critic_prompt = UNGROUNDED_CRITIC_PROMPT.format(draft=draft)
                role = "critic_ungrounded"
            sp.set_inputs({"prompt": critic_prompt, "draft": draft,
                           "lint_report": report["raw"], "grounded": grounded,
                           "params": {"model": MODEL, "temperature": TEMPERATURE}})
            critique_raw = llm(critic_prompt, role=role)
            critique = extract_json(critique_raw)
            sp.set_attribute("n_issues", len(critique.get("issues", [])))
            sp.set_attribute("critic_done", bool(critique.get("done")))
            sp.set_attribute("grounded", grounded)
            sp.set_outputs({"critique": critique, "raw": critique_raw,
                            "n_issues": len(critique.get("issues", [])),
                            "done": bool(critique.get("done"))})

        print(f"\n--- Round {r} critique ({'grounded' if grounded else 'ungrounded'}) ---")
        print(json.dumps(critique, indent=2)[:600])

        if critique.get("done") and n_err_before == 0:
            exit_reason = "critic_and_verifier_agree"
            break
        # Ungrounded critic that bails immediately — surface the failure mode.
        if not grounded and critique.get("done") and not critique.get("issues"):
            print("  [ungrounded critic bailed — no external signal to keep it honest]")
            exit_reason = "ungrounded_bailed"
            break

        # ---- refine ----
        # DO NOT cache: each refine consumes the previous draft + this round's critique.
        with mlflow.start_span(name=f"round_{r}_refine", span_type=SpanType.LLM) as sp:
            refine_prompt = REFINE_PROMPT.format(task=task, draft=draft, critique=json.dumps(critique))
            sp.set_inputs({"prompt": refine_prompt, "previous_draft": draft,
                           "critique": critique,
                           "params": {"model": MODEL, "temperature": TEMPERATURE}})
            new_draft_raw = llm(refine_prompt, role="refine")
            new_draft = extract_code(new_draft_raw)
            new_report = lint(new_draft)
            n_err_after = new_report["n_err"]
            delta = n_err_before - n_err_after
            sp.set_attribute("n_err_before", n_err_before)
            sp.set_attribute("n_err_after", n_err_after)
            sp.set_attribute("delta", delta)
            sp.set_outputs({"new_draft": new_draft, "raw": new_draft_raw,
                            "n_err_before": n_err_before, "n_err_after": n_err_after,
                            "delta": delta, "findings": new_report["findings"]})

        print(f"\n--- Round {r} refined draft ---")
        print(new_draft)
        print(f"\n[verifier] {n_err_after} finding(s) (delta = {delta:+d}):")
        for f in new_report["findings"]:
            print("  •", f)

        history.append({"round": r, "draft": new_draft, "n_err": n_err_after,
                        "findings": list(new_report["findings"])})

        # Decide whether to continue
        if n_err_after == 0:
            draft, report = new_draft, new_report
            exit_reason = "converged"
            break
        if n_err_after >= n_err_before:
            exit_reason = "plateau_or_regression"
            break
        draft, report = new_draft, new_report

    final_n_err = history[-1]["n_err"]
    converged = final_n_err == 0

    mlflow.update_current_trace(tags={
        "rounds_used":  str(len(history) - 1),
        "final_n_err":  str(final_n_err),
        "converged":    str(converged),
        "exit_reason":  exit_reason,
        "variant":      "grounded" if grounded else "ungrounded",
    })

    result = {
        "final_draft": draft,
        "history": history,
        "converged": converged,
        "exit_reason": exit_reason,
        "rounds_used": len(history) - 1,
    }
    if root_span:
        root_span.set_outputs({"final_draft": draft,
                               "converged": converged,
                               "exit_reason": exit_reason,
                               "rounds_used": result["rounds_used"],
                               "final_n_err": final_n_err})
    return result


print("self_refine ready.")

## Round 1: watch the model improve as ruff yells at it

Running the **grounded** variant. Watch each round: draft → ruff/mypy findings verbatim → critic issues JSON → refined draft. The interesting moment is round 1 — the linter calls out the bare except, the missing type hints, the `print`, the `Z` handling, and the critic translates all of that into specific JSON fixes the refiner can apply line-by-line.

In [ ]:
# -- Run the grounded loop ------------------------------------------------

grounded_result = self_refine(TASK, RUBRIC, max_rounds=3, grounded=True)

print("\n" + "=" * 60)
print(f"GROUNDED  | converged={grounded_result['converged']}  "
      f"rounds={grounded_result['rounds_used']}  "
      f"exit={grounded_result['exit_reason']}")
print("=" * 60)

## Open the MLflow UI

1. Visit **http://127.0.0.1:5000** and pick experiment **`session6/demo_06_self_refine_lint`** → Traces tab.
2. Click into the latest **`self_refine_lint`** trace. You'll see the span tree:
   - `self_refine_lint` (CHAIN) at the root
   - `round_0_gen` (LLM)
   - `round_1_critic` (LLM) — attribute `n_issues = 5`, `critic_done = false`
   - `round_1_refine` (LLM) — attributes `n_err_before`, `n_err_after`, `delta`
   - autoLogged OpenAI calls nested under each
3. **Tags** on the trace: `variant`, `converged`, `final_n_err`, `exit_reason`, `rounds_used`. All searchable.
4. After running both variants below, you can filter past traces with `tags.exit_reason = 'converged'` vs `tags.exit_reason = 'ungrounded_bailed'`.

## Now the ungrounded variant — Huang's failure mode

Same loop. Same generator. Same refiner. **Different critic.** This critic does NOT see the lint output. It only sees the draft and is asked "do you see any bugs?"

Expected behaviour:
- Best case: it bails immediately with `"done": true` ("looks fine!"), exiting at round 1 with the broken code still in place.
- Worse case: it invents nitpicks, the refiner "fixes" them, and the linter count *increases*.

Either way, the conclusion is the same: without the external signal, the loop has nothing to converge on.

In [ ]:
# -- Run the ungrounded loop (Huang et al. 2310.01798 failure mode) ------

ungrounded_result = self_refine(TASK, RUBRIC, max_rounds=3, grounded=False)

print("\n" + "=" * 60)
print(f"UNGROUNDED | converged={ungrounded_result['converged']}  "
      f"rounds={ungrounded_result['rounds_used']}  "
      f"exit={ungrounded_result['exit_reason']}")
print("=" * 60)

In [ ]:
# -- Side-by-side comparison ----------------------------------------------

rows = []
for label, result in [("grounded (ruff+mypy)", grounded_result), ("ungrounded", ungrounded_result)]:
    initial = result["history"][0]["n_err"]
    final = result["history"][-1]["n_err"]
    rows.append({
        "variant": label,
        "rounds": result["rounds_used"],
        "initial_n_err": initial,
        "final_n_err": final,
        "delta": initial - final,
        "converged": result["converged"],
        "exit_reason": result["exit_reason"],
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
print()
print("The grounded variant uses linter output the generator literally did not see.")
print("The ungrounded variant has the same model judging its own output — Huang's failure case.")

## The lesson

Huang et al. ([arXiv:2310.01798](https://arxiv.org/abs/2310.01798)) were right: **self-correction without an external signal degrades performance.** The ungrounded loop either bails ("looks fine!") or invents nitpicks that the refiner then introduces new bugs to address.

ruff + mypy work as the external signal for a specific reason: **they catch what the generator literally does not see.** The generator can't run static analysis on its own output during decoding. It can't trace the type of `s.replace("Z", ...)` through `datetime.fromisoformat`. Linters can. So when the critic prompt embeds the linter output, the critic suddenly has information the generator never had — the exact condition Madaan et al. require for Self-Refine to work.

This is **not a property of code-gen.** It's a property of having a verifier. The same pattern works any time you have a structured, reactable error signal.

## Other external signals you can wire up

Anything that returns a structured, reactable error counts. A non-exhaustive list:

| Signal | Catches |
|---|---|
| **`pytest` / unit tests** | Behavioural bugs the linter can't see (wrong logic, off-by-one) |
| **`hypothesis` property tests** | Edge cases the human didn't think of |
| **Pydantic / JSON Schema validators** | Structured-output drift, missing fields, wrong types |
| **`bandit` / `semgrep`** | Security smells (hardcoded creds, SQL injection patterns) |
| **`pylsp` / language-server diagnostics** | Cross-file type errors, unresolved imports |
| **Real DB `EXPLAIN` on generated SQL** | Query plan badness, missing indexes |
| **Docker build / `terraform plan`** | Config drift, broken infra-as-code |
| **Compiler / `tsc` / `rustc`** | The original external verifier — let it scream |

Pattern: pick whichever verifier already exists in your CI. Wire its output into the critic prompt. The Self-Refine loop is the same — only the `lint()` function changes.

## Takeaways

- **Grounded Self-Refine works. Ungrounded Self-Refine does not.** Huang et al. 2310.01798 is the citation; this notebook is the demo.
- **Cap rounds at 2.** Most gains land in round 1 or 2. Round 3+ regression is real (Madaan et al. Fig 4) — exit on `n_err == 0`, `plateau`, or `regression`.
- **Log convergence to MLflow.** Per-round `n_err_before` / `n_err_after` / `delta` attributes turn the loop from "vibes" into a queryable artefact. `tags.exit_reason` lets you filter populations of failures.
- **The verifier IS the technique.** Self-Refine is a thin wrapper around the loop. The intelligence is in the lint/test/schema/exec signal you wire in. No verifier, no technique.

## ⚠️ Reasoning model caveat

This notebook assumes a **non-reasoning** base model. On `o1` / `o3` / Claude Extended Thinking / Gemini Thinking:
- Drop the CoT / ToT / Plan-and-Solve scaffolding — the model does it internally
- Use `developer` role instead of `system` on `o1-2024-12-17+`
- Never pass OpenAI `reasoning_effort` and Gemini `thinking_level` through the same OpenAI-compatible adapter
- See Block 5 of the slides for what's redundant vs what survives

Self-Refine **still works on reasoning models** — the loop's intelligence comes from `ruff + mypy` (the *external* verifier), not from CoT scaffolding in the prompt. The reasoning model handles the refinement reasoning internally; you just hand it the diff between current draft and verifier output. **Never cache** intermediate rounds on a reasoning model either — same rule, same reason.

## Replay every input + output in MLflow

Every prompt and every response from this demo is now in MLflow. Open the UI and click any trace to see the full I/O log.

In [ ]:
import IPython.display as D

ui = MLFLOW_TRACKING_URI.rstrip("/")
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
exp_id = exp.experiment_id if exp else "0"
link = f"{ui}/#/experiments/{exp_id}?searchFilter=&orderByKey=attributes.start_time&orderByAsc=false"
D.display(D.Markdown(f"**Open the experiment in MLflow:** [{link}]({link})"))

recent = mlflow.search_traces(experiment_ids=[exp_id], max_results=5)
recent[["request_id", "timestamp_ms", "execution_time_ms", "tags"]] if not recent.empty else "No traces yet — run the cells above."